# 02. フーリエ解析（Colab）

先に `01_train.ipynb` を実行する。

In [ ]:
# セットアップ：Drive のマウント、リポジトリの取得、runs/ を Drive に向ける（再実行しても壊れない）
import os
import subprocess
import sys

from google.colab import drive

REPO_URL = "https://github.com/karenmathtoml-oss/grokking-from-scratch.git"   # ← 自分のものに書き換える
REPO_DIR = "/content/grokking-from-scratch"
RUNS_DIR = "/content/drive/MyDrive/grokking-runs"


def sh(*cmd):
    """コマンドを実行して、出力をノートブックに表示する。失敗したら例外。"""
    result = subprocess.run(cmd, capture_output=True, text=True)
    print((result.stdout + result.stderr).strip())
    result.check_returncode()


drive.mount("/content/drive")

if os.path.isdir(REPO_DIR):
    sh("git", "-C", REPO_DIR, "pull")
else:
    sh("git", "clone", REPO_URL, REPO_DIR)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

os.makedirs(RUNS_DIR, exist_ok=True)
if not os.path.lexists("runs"):
    os.symlink(RUNS_DIR, "runs")        # runs/ の中身は Drive に残る

sh("pip", "install", "-q", "pytest", "imageio")
sh("git", "log", "-1", "--oneline")     # この実行が、どのコミットのコードかを記録する

## 学習後のスペクトル（ステップ 40000）

In [ ]:
import torch

from grokking.analysis.fourier import embedding_spectrum, neuron_logit_spectrum, top_frequencies
from grokking.config import Config
from grokking.model import Transformer

cfg = Config.load("runs/base/config.json")


def load_model(step: int) -> Transformer:
    """指定したステップのチェックポイントからモデルを復元する。"""
    model = Transformer(cfg.p, cfg.d_model, cfg.n_heads, cfg.d_mlp, seq_len=3, init_seed=cfg.init_seed)
    model.load_state_dict(torch.load(f"runs/base/ckpt_{step:06d}.pt", map_location="cpu"))
    return model


model = load_model(cfg.steps)
spec_E = embedding_spectrum(model.W_E, cfg.p)
spec_L = neuron_logit_spectrum(model.W_U, model.W_out, cfg.p)
print("W_E の上位周波数:", top_frequencies(spec_E))
print("W_L の上位周波数:", top_frequencies(spec_L))

## 学習前（ステップ 0）との比較

学習前のスペクトルは、特定の周波数に偏らないはずです。学習後と並べて違いを確かめます。

In [ ]:
model0 = load_model(0)
spec_E0 = embedding_spectrum(model0.W_E, cfg.p)
print("ステップ 0 の上位周波数:", top_frequencies(spec_E0))
print("最大の強さ（0 / 40000）:", spec_E0.max().item(), "/", spec_E.max().item())

## 学習曲線の確認

In [ ]:
import pandas as pd

df = pd.read_csv("runs/base/metrics.csv")
print(df[df.test_accuracy > 0.5].head(1))
print(df.tail(1))

## 図

工程6で `grokking/analysis/plots.py` を書いたら、図1〜3を作るセルをここに足します。